In [ ]:
!pip install transformers datasets
# !pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import Dataset

# Contoh DataFrame
data = {
    "product": [
        "Smartphone dengan kamera 108MP",
        "Baju kaos katun warna hitam",
        "Buku novel karya Tere Liye",
        "Laptop gaming dengan RAM 16GB",
        "Sepatu lari Nike",
        "Buku resep masakan"
    ],
    "category": [
        "Electronics",
        "Fashion",
        "Books",
        "Electronics",
        "Fashion",
        "Books"
    ]
}

df = pd.DataFrame(data)

# Konversi DataFrame ke Dataset Hugging Face
dataset = Dataset.from_pandas(df)

# Pisahkan dataset menjadi train dan test
dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['product', 'category'],
        num_rows: 4
    })
    test: Dataset({
        features: ['product', 'category'],
        num_rows: 2
    })
})

In [ ]:
from transformers import AutoTokenizer

# Pilih model deepseek-vl2-small
model_name = "deepseek-ai/deepseek-llm-7b-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Jika tokenizer tidak memiliki pad_token, tambahkan atau gunakan eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Gunakan eos_token sebagai pad_token

# Tokenisasi dataset
def tokenize_function(examples):
    return tokenizer(examples["product"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

# Pilih model deepseek-vl2-small dengan lapisan klasifikasi
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(df["category"].unique())  # Jumlah kategori unik
)

config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.97G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at deepseek-ai/deepseek-llm-7b-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import Trainer, TrainingArguments
import torch

# Pastikan model berada di perangkat yang benar (GPU atau CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Persiapan label
label_list = df["category"].unique().tolist()
label_to_id = {label: idx for idx, label in enumerate(label_list)}

def map_labels(examples):
    examples["label"] = label_to_id[examples["category"]]
    return examples

tokenized_datasets = tokenized_datasets.map(map_labels)

# Argumen pelatihan
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",  # Nonaktifkan W&B
)

# Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

# Mulai fine-tuning
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 25870 has 14.73 GiB memory in use. Of the allocated memory 14.63 GiB is allocated by PyTorch, and 1.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)